# PROJET DE TRAITEMENTS DISTRIBUES
Edoardo PICIUCCHI, Aurélien DUVIGNAC-ROSA, Jean-Marc FAUVEL

# Présentation

Le projet consiste en l'étude d'un article intitulé : « CCF : Fast and Scalable Connected Component Computation in MapReduce » (« CCF : Calcul de composantes connexes rapide et scalable en MapReduce »). Cet article a été publié en 2013 par Jimmy Lin et Michael Schatz. Il propose un algorithme de calcul de composantes connexes en utilisant le modèle de programmation MapReduce. Cet algorithme est basé sur l'algorithme de Label Propagation. Il est conçu pour être rapide et scalable, c'est-à-dire qu'il peut être utilisé sur de grands ensembles de données et sur un grand nombre de machines.

L'algorithme a pour objectif d’identifier les sous-graphes connectés au sein d’un graphe G. Le graphe analysé doit être représenté par une collection de paires (N1, N2) où N1 et N2 sont des noeuds du graph G qui ont une connexion entre eux. Le couple (N1, N2) représente une arête du graphe.
L’algorithme présenté dans ce papier va permettre d’identifier des sous-graphes connectés entre eux en modifiant les paires des sous-graphes afin qu’elles soient toutes constituées ainsi : (N, Nmin) où N est un noeud du sous graph et Nmin le noeud de plus petite valeur appartenant au même sous-graphe.
Par ailleurs, l’algorithme ayant pour objet de permettre de traiter des graphes de grande taille, il adopte une programmation distribuée de type MapReduce, permettant ainsi de répartir le traitement sur plusieurs machines et permettre le traitement des données de manière parallèle.

## Objectifs du projet

1. Lire, comprendre et expliquer l’algorithme décrit dans le papier de Hakan Kardes, Siddharth Agrawal, Xin Wang et Ang Sun intitulé « CCF : Fast and Scalable Connected Component Computation in MapReduce » ;
2. Coder l’algorithme en Spark en utilisant à la fois des RDD et des DataFrames ;
3. L’implémentation doit être exécutée en Python, et optionnellement en Scala
4. Effectuer une analyse comparative des versions en RDD et DataFrame sur des graphes de tailles croissantes ;
5. Utilisation de DataBricks pour les petits graphes, et Google Cloud Cluster pour les plus gros (<20 Gb).

## Implémentation de l'algorithme CCF en paradigme Spark

L'implémentation décrite ci-dessous permet de réaliser les étapes de l'algorithme CCF décrites dans l'exemple de l'article.
Les résultats obtenus nous ont permis de constater une erreur au niveau de l'exemple. En effet, dans la figure 5.1 de l'article, le lien entre H et G n'est pas transmis dans le graphe présent dans la colonne *reducer*.

Pour le prouver il suffit d'instancier le paramètre `debug` à `True` dans la méthode `workflow` décrite ci-dessous. Cela permettra d'afficher les graphes successifs obtenus après chaque itération de l'algorithme.

### Importation des librairies

In [1]:
from abc import ABC, abstractmethod
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType

import gzip
import io
import pyspark
import requests
import time

### Création de la classe abstraite Graph

Pour implémenter l'algorithme CCF en Spark, nous avons créé une classe abstraite `Graph` qui permet de représenter un graphe. Cette classe est composée de plusieurs méthodes permettant de mapper, réduire et appliquer l'algorithme CCF sur le graphe.
L'objectif de cette classe est double. D'une part, elle permet de définir les méthodes nécessaires à l'implémentation de l'algorithme CCF. D'autre part, elle assure une meilleure lisibilité du code puisqu'elle permet de définir les méthodes nécessaires à l'implémentation de deux classes filles, respectivement associées aux structures de type *Resilient Distributed Dataset* et *DataFrame*.
En effet, comme présenté dans le code ci-dessous, la méthode `ccf` est définie dans la classe mère mais est appelée dans les classes filles :  permettant ainsi de réduire la redondance du code et d'assurer une meilleure lisibilité.

La représentation d'un graphe se fait par l'intermédiaire de la classe abstraite `Graph`. C'est à partir de celle-ci que deux autres classes, respectivement associées aux structures de type *Resilient Distributed Dataset* et *DataFrame*, seront créées.

La classe `Graph` est composée de trois attributs :
 `sparksession` : la session Spark courante ;
- `edges` : une liste d'éléments de type chaîne de caractères représentant les arêtes du graphe ;
- `numSlices` : un entier représentant le nombre de partitions du graphe (si `numSlices` n'est pas instancié, cette valeur est défini par `pyspark`);

La classe `Graph` est composée de plusieurs méthodes :
- `__init__` : le constructeur de la classe ;
- `__str__` : permet d'afficher le graphe ;
- `format_data` : permet de formater les données du graphe selon que nous utilisions le formation *Resilient Distributed Dataset* (RDD) ou DataFrame (celle-ci est redéfinie dans les classes filles). Remarquons également que cette méthode est instanciée avec le décorateur `@staticmethod` qui permet de l'appeler sans faire appel à une instance de la classe. De plus cette méthode est définie comme une méthode abstraite, c'est-à-dire qu'elle doit être redéfinie dans les classes filles ;
- `mapper` : permet de mapper les noeuds du graphe en créant un graphe bidirectionnel. Cette méthode est instanciée avec le décorateur `@abstractmethod` qui permet de la redéfinir dans les classes filles ;
- `reducer` : permet de réduire le graphe en fusionnant les noeuds connectés. Au même titre que la méthode `mapper`, cette méthode est instanciée avec le décorateur `@abstractmethod` qui permet de la redéfinir dans les classes filles ;
- `ccf` : permet d'appliquer l'algorithme CCF sur le graphe. Cette méthode est définie directement dans la classe mère sans être redéfinie dans les classes filles. Elle permet d'appeler les méthodes `mapper` et `reducer`, redéfinies dans les classes files, pour chaque itération de l'algorithme CCF ;
- `workflow` : permet de définir le workflow de l'algorithme CCF. Cette méthode est définie directement dans la classe mère sans être redéfinie dans les classes filles. Elle permet d'appeler la méthode `ccf` pour chaque itération de l'algorithme CCF et d'afficher le graphe obtenu à chaque itération dans le cas où la valeur `debug` est instanciée à `True`.

**Remarque relative à la méthode `ccf`**

Cette méthode implémente fidélement l'algorithme CCF décrit dans l'article. Elle est composée de deux boucles imbriquées. La première boucle permet de mapper les noeuds du graphe. La seconde boucle permet de réduire les noeuds connectés.
Cependant étant donné que Pyspark s'exécute de manière *lazy*, c'est-à-dire qu'il ne calcule les transformations que lorsqu'une action est appelée, nous avons ajouté une action `count` à l'issue des transformations `map` et `reduce` pour forcer l'exécution de ces transformations.

In [2]:
class Graph(ABC):

  def __init__(self, spark_session, edges, numSlices=None):
    self.spark_session = spark_session
    self.spark_context = self.spark_session.sparkContext
    self.graph = None
    self.data_type = None

  @staticmethod
  def parallelize_edges(spark_context, edges, numSlices=None):
    """
    Convert a list of edges into a distributed RDD.

    Parameters:
      edges (list): List of graph edges as strings.
      numSlices (int): Number of partitions for the RDD.
      if None, use the default number of partitions defined by Spark.

    Returns:
      RDD: Distributed RDD containing the edges.
    """
    return spark_context.parallelize(edges, numSlices)

  @staticmethod
  def format_data(data):
    pass # méthode abstraite, doit être implémentée par les classes filles

  def __str__(self):
    return self.format_data(self.graph)

  @abstractmethod
  def mapper(self):
    pass # méthode abstraite, doit être implémentée par les classes filles

  @abstractmethod
  def reducer(self):
    pass # méthode abstraite, doit être implémentée par les classes filles

  def ccf(self, debug=False):
    nb_iteration = 0
    data = self.graph
    # Initialize accumulator here to be task-specific
    nb_new_pair = self.spark_context.accumulator(0)
    previous_pair = nb_new_pair.value
    current_nb_new_pair = -1
    if debug:
      print(f"iteration {nb_iteration} \n{self}")
    while current_nb_new_pair != 0:
      nb_iteration += 1
      # Perform mapping and reduction steps iteratively
      data_mapper = self.mapper(data)
      data_reducer = self.reducer(data_mapper, nb_new_pair)
      # Remove duplicate edges to reduce redundancy
      data_reducer_distinct = data_reducer.distinct()
      data = data_reducer_distinct

      # Déclenche l'exécution sans ramener les données
      # Cette commande est due à l'évaluation paresseuse de PySpark
      # count() permet de déclencher l'exécution des transformations sans
      # collecter toutes les données sur le driver (ce qui est plus léger que
      # collect()).
      data_reducer_distinct.count()

      current_nb_new_pair = nb_new_pair.value - previous_pair

      if debug :
        data_str_mapper = self.format_data(data_mapper)
        data_str_reducer = self.format_data(data_reducer)
        data_str_reducer_distinct = self.format_data(data_reducer_distinct)
        print(f"Iteration {nb_iteration} \nMapper : \n{data_str_mapper} \nReducer : \n{data_str_reducer} \nReducer.distinct() : \n{data_str_reducer_distinct} \nNew pairs : {current_nb_new_pair}")

      previous_pair = nb_new_pair.value

    return data

  @staticmethod
  def fetch(data):
    pass # méthode abstraite, doit être implémentée par les classes filles

  @staticmethod
  def get_connected_component_count(data):
    pass # méthode abstraite, doit être implémentée par les classes filles

  def workflow(self, debug=False):
    data = self.ccf(debug=debug)
    final_data_str = self.format_data(data)
    if debug:
      print(f"{self.data_type} obtenue à l'issue de l'algorithme CCF: \n{final_data_str}")
      # Extraction du résultat final (fetch)
      print(f"Final Results (Node -> Component): \n{self.fetch(data)}")
    # Print the number of connected components
    print(f"\nNumber of connected components: {self.get_connected_component_count(data)}")

### Création de la classe fille GraphRDD

In [3]:
class GraphRDD(Graph):
  def __init__(self, spark_session, edges, numSlices=None):
    """
    Initialize the GraphRDD with a list of edges.
    """
    super().__init__(spark_session, edges, numSlices)
    self.graph = self.parallelize_edges(self.spark_context, edges, numSlices).map(lambda x: x.split(" ")).map(lambda x: (x[0], x[1]))
    self.data_type = "RDD"

  @staticmethod
  def format_data(data):
    sorted_rows = sorted(data.collect(), key=lambda x: (x[0], x[1]))
    formatted_rows = [f"{source} -> {destination}" for source, destination in sorted_rows]
    formatted_rows = "\n".join(formatted_rows)
    return formatted_rows

  @staticmethod
  def mapper(rdd):
    return rdd.flatMap(lambda x: [x, (x[1], x[0])])

  def reducer(self, data, nb_new_pair):
    """
    Perform the reduce step of the algorithm and propagate the smallest component ID for Resilient Distributed Dataset
    """
    def count_nb_new_pair(x):
        """
        Count new pairs and propagate smallest ID
        """
        key, values = x
        min = key
        value_list = []
        for value in values:
            if value < min:
                min = value
            value_list.append(value)
        if min < key:
            yield(key, min)
            for value in value_list:
                if value != min:
                    nb_new_pair.add(1)
                    yield(value, min)
    # Group edges by source node and propagate smallest component ID
    return data.groupByKey().flatMap(lambda x: count_nb_new_pair(x))

  @staticmethod
  def fetch(data):
    output_str = ""
    data = data.map(lambda x: (x[0], x[1])).collect()
    for node, component in sorted(data):
      output_str += f"Node {node} belongs to Component {component}\n"
    return output_str

  @staticmethod
  def get_connected_component_count(data):
    return data.map(lambda x: (x[0], x[1])).distinct().count()

### Création de la classe fille GraphDF

In [4]:
class GraphDF(Graph):
  def __init__(self, spark_session, edges, numSlices=None):
    """
    Initialize the GraphDF with a list of edges.
    """
    super().__init__(spark_session, edges, numSlices)
    # The list of edges is parallelized into an RDD.
    # The RDD is then converted into a DataFrame with a single column "raw" using createDataFrame.
    df = self.spark_session.createDataFrame(self.parallelize_edges(self.spark_context, edges, numSlices).map(lambda x: (x,)), ["raw"])
    self.graph = df.withColumn("k", split(df["raw"], " ").getItem(0)) \
                  .withColumn('v', split(df["raw"], " ").getItem(1)) \
                  .drop("raw")
    self.data_type = "DataFrame"

  @staticmethod
  def format_data(df):
    rows = df.selectExpr("k as source", "v as destination").collect()
    sorted_rows = sorted(rows, key=lambda x: (x[0], x[1]))
    formatted_rows = [f"{row['source']} -> {row['destination']}" for row in sorted_rows]
    # Join all formatted edges with newlines
    return "\n".join(formatted_rows)

  @staticmethod
  def mapper(data):
    return data.union(data.select(col("v").alias("k"), col("k").alias("v")))

  def reducer(self, data, nb_new_pair):
    """
    Perform the reduce step of the algorithm and propagate the smallest component ID for DateFrame
    """
    # Group by source node, propagate smallest component ID, and count new pairs
    data = (data
        .groupBy("k")
        .agg(
            collect_set("v").alias("v"),  # Collect all values in a set
        )
        .withColumn("min", least(col("k"), array_min(col("v"))))  # Calculate "min" directly
        .filter(col("k") != col("min"))  # Filter rows where "k" != "min"
    )

    # Count the number of new pairs added in this iteration
    nb_new_pair += data.withColumn("count", size("v") - 1).select(sum("count")).collect()[0][0]

    data = data.select(
      col("min").alias("a_min"),
      expr("filter(concat(array(k), v), x -> x != min)").alias("valueList")
    )
    data = data.select(
      explode(col("valueList")).alias("k"),  # Décomposer `valueList` en une ligne par élément
      col("a_min").alias("v")  # Conserver `a_min`
    )

    return data

  @staticmethod
  def fetch(data):
    output_str = ""
    data = data.select("k", "v").distinct().collect()
    for row in sorted(data, key=lambda x: x["k"]):
      output_str += f"Node {row['k']} belongs to Component {row['v']}\n"
    return output_str

  @staticmethod
  def get_connected_component_count(data):
    return data.select('k').distinct().count()

### Exécution de l'algorithme CCF avec les instances de GraphRDD et GraphDF

Dans l'exemple ci-dessous, nous avons créé le graphe issu de l'article proposé. Nous avons ensuite appliqué l'algorithme CCF sur ce graphe en utilisant les instances de `GraphRDD` et `GraphDF`. Nous avons affiché le graphe obtenu à chaque itération de l'algorithme à l'aide de l'attribut `debug` instancié à `True`.

Pour rappel, le graphe d'exemple est donné ci-après :

<img src="figures/graphe_exemple.png" alt="Description de l'image" style="width:600px;">

In [5]:
# List of edges in the graph
edges = [
  "A B",
  "B C",
  "B D",
  "D E",
  "F G",
  "G H"
]
numSlices = None
debug = False
print("SparkSession opening.")
spark_session = SparkSession \
  .builder \
  .appName("PySpark Hardcoded Graph") \
  .getOrCreate()
# Create GraphRDD and GraphDF instances
RDD_Demo = GraphRDD(spark_session, edges, numSlices=numSlices)
DF_Demo = GraphDF(spark_session, edges, numSlices=numSlices)
# Execute CCF algorithm on RDD instance
start_time = time.time()
RDD_Demo.workflow(debug)
end_time = time.time()
print(f"Temps d'exécution CCF sur RDD: {end_time - start_time:.6f} secondes")
# Execute CCF algorithm on DF instance
start_time = time.time()
DF_Demo.workflow(debug)
end_time = time.time()
print(f"Temps d'exécution CCF sur DateFrame: {end_time - start_time:.6f} secondes")
# spark_session.stop() peut poser des problèmes dans databricks
spark_session.stop()
print("SparkSession closed.")

SparkSession opening.

Number of connected components: 6
Temps d'exécution CCF sur RDD: 9.541420 secondes

Number of connected components: 6
Temps d'exécution CCF sur DateFrame: 42.226003 secondes
SparkSession closed.


In [6]:
def fetch_and_process_file(url):
    """
    Télécharge un fichier compressé à partir d'une URL, le décompresse, et traite les lignes.
    Les lignes commençant par '#' sont exclues, et les séparateurs multiples sont remplacés par un espace unique.

    Args:
        url (str): L'URL du fichier compressé à télécharger.

    Returns:
        list: Une liste de chaînes représentant les lignes traitées.
    """
    # Télécharger le fichier
    response = requests.get(url)
    compressed_file = io.BytesIO(response.content)  # Charger le contenu dans un objet en mémoire
    decompressed_file = gzip.GzipFile(fileobj=compressed_file)  # Décompresser

    # Lire les lignes du fichier
    lines = decompressed_file.read().decode('utf-8').splitlines()

    # Traiter les lignes : supprimer les commentaires et standardiser les séparateurs
    processed_lines = [
        " ".join(line.split())  # Remplacer les espaces multiples par un seul espace
        for line in lines if not line.startswith("#")  # Exclure les lignes de commentaires
    ]

    return processed_lines

In [ ]:
# URL du fichier
urls = [
    "https://snap.stanford.edu/data/email-Eu-core.txt.gz",
    "https://snap.stanford.edu/data/email-Enron.txt.gz"]

for url in urls:
    print("SparkSession opening.")
    spark_session = SparkSession \
        .builder \
        .appName("PySpark Hardcoded Graph") \
        .getOrCreate()
    edges = fetch_and_process_file(url)
    numSlices = None
    debug = False
    # Create GraphRDD and GraphDF instances
    RDD = GraphRDD(spark_session, edges, numSlices=numSlices)
    DF = GraphDF(spark_session, edges, numSlices=numSlices)
    # Execute CCF algorithm on RDD instance
    start_time = time.time()
    RDD.workflow(debug)
    end_time = time.time()
    print(f"Temps d'exécution CCF sur RDD: {end_time - start_time:.6f} secondes")
    # Execute CCF algorithm on DF instance
    start_time = time.time()
    DF.workflow(debug)
    end_time = time.time()
    print(f"Temps d'exécution CCF sur DateFrame: {end_time - start_time:.6f} secondes")
    # spark_session.stop() peut poser des problèmes dans databricks
    spark_session.stop()
    print("SparkSession closed.")

SparkSession opening.

Number of connected components: 985
Temps d'exécution CCF sur RDD: 32.233084 secondes

Number of connected components: 985
Temps d'exécution CCF sur DateFrame: 72.896196 secondes
SparkSession closed.
SparkSession opening.

Number of connected components: 35627
Temps d'exécution CCF sur RDD: 19.346496 secondes
